# Noise shape characterization — the hyperbolic $\xi$–$\chi$ triangle

HyperWave's **hyperbolic likelihood** is a heavy-tailed generalisation of the Gaussian/Whittle
likelihood. Instead of assuming the whitened noise is Normal, it fits a *shape* per frequency
segment through two parameters $(\alpha,\delta)$, which map onto the **generalised-hyperbolic
shape triangle**:

* bottom vertex ($\xi\to 0$): the **Normal** distribution (Gaussian noise),
* top edge ($\xi\to 1$): the **skew-Laplace** limit (heavy tails),
* horizontal axis $\chi$: skewness (only exercised by the asymmetric model; the symmetric
  filter used here keeps $\chi=0$, so distributions separate **vertically** by tail weight).

The triangle height is $\xi = (1+\alpha\delta)^{-1/2}$ (with $\beta=0$). We generate three
whitened-noise streams — **Gaussian**, **Student-t**, **inverse-Gaussian** — characterise each
with `DataInference`, and check they land where they should.

In [1]:
import os, warnings
import numpy as np
import bilby
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import invgauss

from hyperwave.likelihoods import LogLike
from hyperwave.inference import DataInference
from hyperwave.plots.hyper import Shape   # provides the xi_ksi triangle map

OUT = 'results/noise'
os.makedirs(os.path.join(OUT, 'chains'), exist_ok=True)
DF = 0.25
FREQS = np.arange(20.0, 512.0, DF)
NF = FREQS.size
NSEG = 4
print(f'{NF} frequency bins, {NSEG} shape segments')

1968 frequency bins, 4 shape segments


## 1. Generate non-Gaussian noise

Each stream is whitened (unit-variance) complex frequency-domain noise; only the **amplitude
law** differs. Student-t ($\nu=3$) has heavier tails than Gaussian; inverse-Gaussian is
positively skewed in power.

In [2]:
def make_noise(dist, n, rng):
    if dist == 'gaussian':
        return (rng.standard_normal(n) + 1j * rng.standard_normal(n)) / np.sqrt(2)
    if dist == 'student_t':
        nu = 3
        scale = np.sqrt(2 * nu / (nu - 2))
        return (rng.standard_t(nu, n) + 1j * rng.standard_t(nu, n)) / scale
    if dist == 'inverse_gaussian':
        # Normal-inverse-Gaussian (NIG): a Normal variance mixture x = sqrt(W) Z
        # with W ~ Inverse-Gaussian -> genuinely HEAVY-tailed. (The earlier
        # sqrt(IG)-amplitude form was light-tailed and mischaracterised as Normal.)
        W = invgauss.rvs(0.15, size=n, random_state=rng)
        W /= np.mean(W)
        return (np.sqrt(W) * rng.standard_normal(n)
                + 1j * np.sqrt(W) * rng.standard_normal(n)) / np.sqrt(2)
    raise ValueError(dist)

DISTS = ['gaussian', 'student_t', 'inverse_gaussian']

## 2. Characterise the noise shape

`LogLike(..., ddims=True, nsegs=NSEG)` is the data-only hyperbolic likelihood; its per-segment
$(\log_{10}\alpha,\ \log_{10}\delta/\alpha)$ are sampled as nuisance parameters with Eryn
(parallel-tempered). We take the posterior-median $(\alpha,\delta)$ per segment.

In [3]:
def characterize(data, tag, nsegs=NSEG, burn=1000, nsteps=2000):
    like = LogLike(data, FREQS, ['X'], ddims=True, nsegs=nsegs, freq_domain=True)
    npr = {}
    for i in range(nsegs):
        npr[f'log_alpha_{i}'] = bilby.core.prior.Uniform(-2, 3)
    for i in range(nsegs):
        npr[f'log_ratio_{i}'] = bilby.core.prior.Uniform(-2, 3)
    h5 = os.path.join(OUT, 'chains', f'{tag}_eryn.h5')
    if os.path.exists(h5):
        os.remove(h5)
    inf = DataInference(like.hyperbolic, noise_priors=npr,
                        common_params={'save_dir': OUT, 'TAG': tag},
                        sampler_kwargs=dict(burn=burn, nsteps=nsteps), sampler_name='eryn')
    inf.run()
    return inf.get_samples()

shapes = {}
for dist in DISTS:
    samples = characterize(make_noise(dist, NF, np.random.default_rng(0)), tag=f'shape_{dist}')
    alpha = 10 ** np.median(samples[:, :NSEG], axis=0)
    delta = alpha * 10 ** np.median(samples[:, NSEG:], axis=0)
    shapes[dist] = (alpha, delta)
    _, ksi, _ = Shape.xi_ksi(beta=0.0, alpha=alpha, delta=delta)
    print(f'{dist:18s}  median xi (triangle height) = {np.median(ksi):.3f}')

nwalkers: 16
ntemps: 5
burn: 1000
nsteps: 2000
> Running Eryn MCMC...


  0%|          | 0/1000 [00:00<?, ?it/s]

  1%|▏         | 14/1000 [00:00<00:07, 136.13it/s]

  3%|▎         | 30/1000 [00:00<00:06, 146.98it/s]

  5%|▍         | 46/1000 [00:00<00:06, 149.37it/s]

...

100%|█████████▉| 1997/2000 [02:13<00:00, 14.71it/s]

100%|█████████▉| 1999/2000 [02:13<00:00, 15.30it/s]

100%|██████████| 2000/2000 [02:13<00:00, 14.98it/s]

> Eryn sampling finished. Output at: results/noise/chains/shape_inverse_gaussian_eryn.h5
Loading samples from results/noise/chains/shape_inverse_gaussian_eryn.h5


16000
inverse_gaussian    median xi (triangle height) = 0.234


## 3. The shape triangle

Each star is one frequency segment. **Gaussian sits at the Normal vertex (bottom)**, **Student-t
climbs toward the skew-Laplace edge (top)**, and **inverse-Gaussian lands in between** — its
heavier-than-Normal power tail pushing it up. This is the visual proof that the hyperbolic
filter recovers the correct noise shape.

In [4]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([-1, 1, 0, -1], [1, 1, 0, 1], '-', color='k', lw=1.2)
ax.text(0.0, 1.05, 'skew-Laplace', ha='center', style='italic', fontsize=11)
ax.text(0.0, -0.08, 'Normal', ha='center', style='italic', fontsize=11)
colors = {'gaussian': '#0f9b8e', 'student_t': '#ca0147', 'inverse_gaussian': '#f2ab15'}
for dist in DISTS:
    alpha, delta = shapes[dist]
    xi, ksi, _ = Shape.xi_ksi(beta=0.0, alpha=alpha, delta=delta)
    ax.plot(xi, ksi, '*', ms=15, color=colors[dist], alpha=0.75, label=dist.replace('_', '-'))
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-0.17, 1.17)
ax.set_xlabel(r'$\chi$  (skewness)')
ax.set_ylabel(r'$\xi$  (tail weight)')
ax.set_title('Hyperbolic noise shape triangle')
ax.legend(loc='center right', frameon=True)
fig.tight_layout()
out = os.path.join(OUT, 'shape_triangle.png')
fig.savefig(out, dpi=150, bbox_inches='tight')
print('saved ->', out)

saved -> results/noise/shape_triangle.png


## Takeaway

The separation (Gaussian $\xi\approx0$, Student-t $\xi\approx0.9$, inverse-Gaussian in between)
shows the hyperbolic likelihood **reads off the noise distribution shape correctly**. In real GW
data the same machinery flags non-Gaussian/glitchy segments and down-weights them — the
robustness the `--like hyperbolic` path buys over a plain Gaussian likelihood.